In [ ]:
import os, gc, shutil, subprocess, tempfile
from pathlib import Path
from datetime import datetime

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from tqdm.auto import tqdm

from IPython.display import display, Image as IPyImage, Video

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} \nVRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
SD_CKPT = Path(r"D:\AI\sd.webui\webui\models\Stable-diffusion\realisticVisionV60B1_v51HyperVAE.safetensors") # Please replace the path with your own checkpoint path
ADAPTER_CKPT = Path("vjepa_t2i_adapter_sd15_step4550.pt")

OUTPUT_DIR = Path("experiments/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 384

In [ ]:
vjepa_transform = T.Compose([
    T.Resize(IMG_SIZE, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# def letterbox(img, size=IMG_SIZE):
#     w, h = img.size
#     scale = size / max(w, h)
#     nw, nh = int(round(w * scale)), int(round(h * scale))
#     img = img.resize((nw, nh), Image.BICUBIC)
#     out = Image.new("RGB", (size, size), (0, 0, 0))
#     out.paste(img, ((size - nw) // 2, (size - nh) // 2))
#     return out

# vjepa_transform = T.Compose([
#     T.Lambda(letterbox),
#     T.ToTensor(),
#     T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
# ])

In [ ]:
class ResnetBlock(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.GroupNorm(32, channels, eps=1e-6), nn.SiLU(),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.GroupNorm(32, channels, eps=1e-6), nn.SiLU(),
            nn.Conv2d(channels, channels, 3, padding=1),
        )
    def forward(self, x): return x + self.block(x)

class AdapterBlock(nn.Module):
    def __init__(self, in_channels, out_channels, num_res_blocks=2):
        super().__init__()
        self.in_conv = nn.Conv2d(in_channels, out_channels, 1)
        self.resnets = nn.Sequential(*[ResnetBlock(out_channels) for _ in range(num_res_blocks)])
    def forward(self, x): return self.resnets(self.in_conv(x))

class DownsampleBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, 3, stride=2, padding=1)
    def forward(self, x): return self.conv(x)

class JEPAProjectionHead(nn.Module):
    def __init__(self, token_dim=1024, out_channels=320):
        super().__init__()
        self.reduce = nn.Conv2d(token_dim, 512, 1)
        self.refine = nn.Sequential(
            nn.GroupNorm(32, 512, eps=1e-6), nn.SiLU(),
            nn.Conv2d(512, out_channels, 3, padding=1),
            nn.GroupNorm(32, out_channels, eps=1e-6), nn.SiLU(),
        )
    def forward(self, x):
        B = x.shape[0]
        x = x.float().reshape(B, 24, 24, 1024).permute(0, 3, 1, 2)
        x = self.reduce(x)
        x = F.interpolate(x, size=(48, 48), mode='bilinear', align_corners=False)
        return self.refine(x)

class JEPAAdapter(nn.Module):
    def __init__(self, num_res_blocks=2):
        super().__init__()
        self.projection = JEPAProjectionHead(1024, 320)
        self.block0 = AdapterBlock(320, 320, num_res_blocks)
        self.block1 = AdapterBlock(320, 640, num_res_blocks)
        self.block2 = AdapterBlock(640, 1280, num_res_blocks)
        self.block3 = AdapterBlock(1280, 1280, num_res_blocks)
        self.down0 = DownsampleBlock(320)
        self.down1 = DownsampleBlock(640)
        self.down2 = DownsampleBlock(1280)

    def forward(self, jepa_emb):
        x = self.projection(jepa_emb)
        f0 = self.block0(x); x = self.down0(f0)
        f1 = self.block1(x); x = self.down1(f1)
        f2 = self.block2(x); x = self.down2(f2)
        f3 = self.block3(x)
        return [f0, f1, f2, f3]

In [ ]:
_model_cache = {
    "vjepa_encoder": None,
    "adapter": None,
    "sd_components": None,
    "ckpt_hash": None,
}

def clear_cache():
    for k in ["vjepa_encoder", "adapter", "sd_components"]:
        if _model_cache[k]:
            if isinstance(_model_cache[k], dict):
                for m in _model_cache[k].values():
                    if hasattr(m, 'cpu'): m.cpu()
            elif hasattr(_model_cache[k], 'cpu'):
                _model_cache[k].cpu()
            _model_cache[k] = None
    gc.collect()
    torch.cuda.empty_cache()
    print("✓ Cache cleared")

In [ ]:
from diffusers import StableDiffusionPipeline, LCMScheduler, DDIMScheduler

def load_vjepa():
    if _model_cache["vjepa_encoder"] is not None:
        return _model_cache["vjepa_encoder"].to(DEVICE)
    print("Loading V-JEPA 2.1...")
    encoder, _ = torch.hub.load('facebookresearch/vjepa2', 'vjepa2_1_vit_large_384')
    encoder.eval().to(DEVICE)
    for p in encoder.parameters(): p.requires_grad_(False)
    _model_cache["vjepa_encoder"] = encoder
    return encoder

def load_sd():
    if _model_cache["sd_components"] and _model_cache["ckpt_hash"] == str(SD_CKPT):
        return _model_cache["sd_components"]

    print(f"Loading SD: {SD_CKPT.name}")
    pipe = StableDiffusionPipeline.from_single_file(
        str(SD_CKPT), torch_dtype=torch.float16, safety_checker=None
    ).to(DEVICE)

    # LCM Scheduler + LoRA, yields faster speeds, good for video generation
    # scheduler = LCMScheduler.from_config(pipe.scheduler.config)
    # pipe.load_lora_weights("latent-consistency/lcm-lora-sdv1-5")
    # pipe.fuse_lora()

    # DDIM Scheduler, yields better quality, good for image generation, but slower
    scheduler = DDIMScheduler.from_config(pipe.scheduler.config)

    comps = {
        "vae": pipe.vae.eval(), "unet": pipe.unet.eval(),
        "tokenizer": pipe.tokenizer, "text_enc": pipe.text_encoder.eval(),
        "scheduler": scheduler
    }
    for m in [comps["vae"], comps["unet"], comps["text_enc"]]:
        for p in m.parameters(): p.requires_grad_(False)

    del pipe; gc.collect(); torch.cuda.empty_cache()
    _model_cache["sd_components"] = comps
    _model_cache["ckpt_hash"] = str(SD_CKPT)
    return comps

def load_adapter():
    if _model_cache["adapter"] is not None:
        return _model_cache["adapter"]
    print(f"Loading Adapter: {ADAPTER_CKPT}")
    adapter = JEPAAdapter(num_res_blocks=2).to(DEVICE)
    ckpt = torch.load(str(ADAPTER_CKPT), map_location=DEVICE, weights_only=True)
    adapter.load_state_dict(ckpt.get("adapter", ckpt))
    adapter.eval()
    _model_cache["adapter"] = adapter
    return adapter

In [ ]:
@torch.no_grad()
def encode_frame(pil_image: Image.Image):
    encoder = load_vjepa()
    x = vjepa_transform(pil_image).unsqueeze(0).unsqueeze(2).to(DEVICE)
    x = x.to(next(encoder.parameters()).dtype)
    with torch.autocast(DEVICE, enabled=True):
        emb = encoder(x)
    return emb.squeeze(0).float().cpu()

def encode_all_frames(frames):
    encoder = load_vjepa()
    embeddings = [encode_frame(f) for f in tqdm(frames, desc="V-JEPA encode")]
    encoder.cpu(); torch.cuda.empty_cache()
    return embeddings

In [ ]:
@torch.no_grad()
def get_text_emb(tokenizer, text_enc, prompt):
    tokens = tokenizer([prompt], padding="max_length",
                      max_length=tokenizer.model_max_length,
                      truncation=True, return_tensors="pt").input_ids.to(DEVICE)
    return text_enc(tokens).last_hidden_state

@torch.no_grad()
def generate_image(jepa_emb, adapter, vae, unet, scheduler, text_emb,
                   cfg_scale=1.0, num_steps=4, seed=42, cond_scale=1.0):
    scheduler.set_timesteps(num_steps)
    gen = torch.Generator(device=DEVICE).manual_seed(seed)
    latent = torch.randn(1,4,48,48, dtype=torch.float16, device=DEVICE, generator=gen)
            
    jepa_gpu = jepa_emb.unsqueeze(0).to(DEVICE)
    use_cfg = cfg_scale > 1.0
    null_emb = torch.zeros_like(text_emb) if use_cfg else None

    for t in scheduler.timesteps:
        feats = adapter(jepa_gpu)
        residuals = [f.to(torch.float16) * cond_scale for f in feats]

        if use_cfg:
            noise_u = unet(latent, t, null_emb, down_intrablock_additional_residuals=residuals).sample
            noise_c = unet(latent, t, text_emb, down_intrablock_additional_residuals=residuals).sample
            noise_pred = noise_u + cfg_scale * (noise_c - noise_u)
        else:
            noise_pred = unet(latent, t, text_emb, down_intrablock_additional_residuals=residuals).sample

        latent = scheduler.step(noise_pred, t, latent).prev_sample

    latent = latent / vae.config.scaling_factor
    img = vae.decode(latent).sample
    img = ((img.float().clamp(-1,1)+1)/2 * 255).byte()
    return Image.fromarray(img[0].permute(1,2,0).cpu().numpy())

In [ ]:
def extract_video_frames(video_path, target_fps=0):
    cap = cv2.VideoCapture(video_path)
    orig_fps = cap.get(cv2.CAP_PROP_FPS) or 24.0
    interval = 1 if target_fps <= 0 else max(1, int(orig_fps/target_fps))
    eff_fps = orig_fps / interval

    frames = []
    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        if idx % interval == 0:
            frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
        idx += 1
    cap.release()
    print(f"Extracted {len(frames)} frames @ {eff_fps:.2f} fps")
    return frames, eff_fps

def create_side_by_side(orig, gen):
    h = max(orig.height, gen.height)
    o = orig.resize((int(orig.width*h/orig.height), h))
    g = gen.resize((int(gen.width*h/gen.height), h))
    w = o.width + g.width + 4
    # Ensure even dimensions for h264
    w += w % 2; h_total = h + 30 + (h+30) % 2
    canvas = Image.new("RGB", (w, h_total), (30,30,30))
    canvas.paste(o, (0,0)); canvas.paste(g, (o.width+4,0))
    return canvas

def frames_to_video(frames, path, fps):
    tmp = tempfile.mkdtemp()
    for i,f in enumerate(frames):
        f.save(f"{tmp}/f_{i:06d}.png")
    cmd = ["ffmpeg","-y","-framerate",str(round(fps,3)),"-i",f"{tmp}/f_%06d.png",
           "-c:v","libx264","-pix_fmt","yuv420p","-crf","23",path]
    subprocess.run(cmd, capture_output=True)
    shutil.rmtree(tmp)

In [ ]:
def run_pipeline(input_path, prompt="", cfg_scale=1.0, steps=4, seed=42, cond_scale=1.0, video_fps=0):
    input_path = Path(input_path)
    is_video = input_path.suffix.lower() in {".mp4",".avi",".mov",".mkv",".webm"}

    # Load frames
    if is_video:
        frames, fps = extract_video_frames(str(input_path), video_fps)
    else:
        frames = [Image.open(input_path).convert("RGB")]
        fps = None

    # Encode
    embeddings = encode_all_frames(frames)

    # Load models
    sd = load_sd(); adapter = load_adapter()
    text_emb = get_text_emb(sd["tokenizer"], sd["text_enc"], prompt)

    # Generate
    generated = []
    for emb in tqdm(embeddings, desc="Generating"):
        img = generate_image(emb, adapter, sd["vae"], sd["unet"], sd["scheduler"],
                            text_emb, cfg_scale, steps, seed, cond_scale)
        generated.append(img)

    # Save side-by-side
    timestamp = datetime.now().strftime("%H%M%S")
    if is_video:
        sbs_frames = [create_side_by_side(o,g) for o,g in zip(frames, generated)]
        out_path = OUTPUT_DIR / f"{input_path.stem}_sbs_{timestamp}.mp4"
        frames_to_video(sbs_frames, str(out_path), fps)
    else:
        sbs = create_side_by_side(frames[0], generated[0])
        out_path = OUTPUT_DIR / f"{input_path.stem}_sbs_{timestamp}.png"
        sbs.save(out_path)

    print(f"✓ Saved: {out_path}")
    return out_path, frames[0] if not is_video else None, generated[0] if not is_video else None

**cond_scale** = Conditioning from the T2I Adapter. keeping at 1.0 yields high faithfulness to the original image, but the resultant image has a poorer quality
**cfg_scale** = Firmly keep at 1. Completely overrides the adapter if >1
**prompt** = Since CFG cannot be >1, therefore this doesn't work.
**steps** = If using LCM + LoRA, set to 4-8, maybe 2 if only doing quick experiments. If using DDIM, 15+ is recommended, but can be somewhat slow for video.

In [ ]:
# Change these parameters
input_image = "experiments/rocks.jpg" # put your image path here
prompt = ""
cfg = 1.0
steps = 20
seed = 42

out_path, orig, gen = run_pipeline(
    input_image,
    prompt=prompt,
    cfg_scale=cfg,
    steps=steps,
    seed=seed,
    cond_scale=1.0 # 1.0 is normal conditioning, but can at times reduce the quality of the output, adjust accordingly to find a balance between quality and faithfulness to the input video
)

# Display result
display(IPyImage(str(out_path)))

In [ ]:
# input_video = "experiments/input.mp4"
# prompt = ""

# out_path, _, _ = run_pipeline(
#     input_video,
#     prompt=prompt,
#     steps=4,
#     video_fps=5, # 0 = original fps, or set to a target fps like 5 or 10 for faster processing
#     cond_scale=0.8
# )

# # Display video
# display(Video(str(out_path), embed=True))

In [ ]:
class TokenCompressor(nn.Module):
    def __init__(self, embed_dim=1024, grid_size=24):
        super().__init__()
        self.grid_size = grid_size
        self.unshuffle = nn.PixelUnshuffle(downscale_factor=2)
        self.network = nn.Sequential(
            nn.Linear(embed_dim * 4, embed_dim * 2),
            nn.SiLU(),
            nn.Linear(embed_dim * 2, embed_dim)
        )

    def forward(self, x):
        B, N, C = x.shape
        x = x.transpose(1, 2).view(B, C, self.grid_size, self.grid_size)
        x = self.unshuffle(x)           # [B, 4096, 12, 12]
        x = x.flatten(2).transpose(1, 2)  # [B, 144, 4096]
        return self.network(x)          # [B, 144, 1024]


class TokenRefiner(nn.Module):
    def __init__(self, embed_dim=1024, target_grid=12):
        super().__init__()
        self.target_grid = target_grid
        self.network = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 2),
            nn.SiLU(),
            nn.Linear(embed_dim * 2, embed_dim * 4)
        )
        self.shuffle = nn.PixelShuffle(upscale_factor=2)

    def forward(self, x):
        B, N, C = x.shape
        x = self.network(x)             # [B, 144, 4096]
        x = x.transpose(1, 2).view(B, C * 4, self.target_grid, self.target_grid)
        refined = self.shuffle(x)       # [B, 1024, 24, 24]
        return refined.flatten(2).transpose(1, 2)  # [B, 576, 1024]

import random
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from pathlib import Path
from PIL import Image
import torch

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}


@torch.no_grad()
def load_compressor_refiner(compressor_ckpt, refiner_ckpt):
    comp = TokenCompressor().to(DEVICE)
    comp.load_state_dict(torch.load(compressor_ckpt, map_location=DEVICE))
    comp.eval()

    ref = TokenRefiner().to(DEVICE)
    ref.load_state_dict(torch.load(refiner_ckpt, map_location=DEVICE))
    ref.eval()

    return comp, ref


@torch.no_grad()
def decode_embedding(emb, adapter, vae, unet, scheduler, text_emb,
                     cond_scale=1.0, num_steps=8, seed=42):
    """
    Decode a single (576, 1024) embedding to a PIL Image at 384x384.
    text_emb must be pre-computed and passed in — no SD reload here.
    """
    # Fresh scheduler state for each decode so the two runs don't share state
    scheduler.set_timesteps(num_steps)

    gen    = torch.Generator(device=DEVICE).manual_seed(seed)
    latent = torch.randn(1, 4, 48, 48, dtype=torch.float16,
                         device=DEVICE, generator=gen)

    emb_gpu = emb.unsqueeze(0).to(DEVICE)

    for t in scheduler.timesteps:
        feats      = adapter(emb_gpu)
        residuals  = [f.to(torch.float16) * cond_scale for f in feats]
        noise_pred = unet(latent, t, text_emb,
                          down_intrablock_additional_residuals=residuals).sample
        latent     = scheduler.step(noise_pred, t, latent).prev_sample

    latent = latent / vae.config.scaling_factor
    img    = vae.decode(latent).sample
    img    = ((img.float().clamp(-1, 1) + 1) / 2 * 255).byte()
    return Image.fromarray(img[0].permute(1, 2, 0).cpu().numpy())


@torch.no_grad()
def run_triple_comparison(
    image_folder,
    image_path,
    compressor_ckpt,
    refiner_ckpt,
    cond_scale,
    num_steps,
    seed,
    save_path,
):
    # ── 1. Pick image ─────────────────────────────────────────────────────────
    if image_path is not None:
        img_path = Path(image_path)
        assert img_path.exists() and img_path.suffix.lower() in IMG_EXTS, \
            f"Image not found or unsupported format: {img_path}"
    else:
        folder = Path(image_folder)
        if folder.is_file():
            img_path = folder
        else:
            candidates = [p for p in folder.iterdir() if p.suffix.lower() in IMG_EXTS]
            assert candidates, f"No images found in {folder}"
            img_path = random.choice(candidates)

    print(f"Selected: {img_path.name}")
    orig_pil = Image.open(img_path).convert("RGB")

    # ── 2. Encode with V-JEPA ─────────────────────────────────────────────────
    print("Encoding with V-JEPA 2.1...")
    emb_orig = encode_frame(orig_pil)   # (576, 1024) float32, CPU
    torch.cuda.empty_cache()

    # ── 3. Compress → refine ──────────────────────────────────────────────────
    print("Compressing + refining embedding...")
    compressor, refiner = load_compressor_refiner(compressor_ckpt, refiner_ckpt)
    emb_gpu        = emb_orig.unsqueeze(0).to(DEVICE)   # (1, 576, 1024)
    compressed     = compressor(emb_gpu)                 # (1, 144, 1024)
    emb_compressed = refiner(compressed).squeeze(0).cpu()  # (576, 1024)
    del compressor, refiner
    torch.cuda.empty_cache()

    # ── 4. Load SD + adapter once, compute null text embed once ───────────────
    print("Loading SD + adapter...")
    sd       = load_sd()
    adapter  = load_adapter()
    vae      = sd["vae"]
    unet     = sd["unet"]
    scheduler = sd["scheduler"]

    # Compute null text embed once and reuse for both decodes
    text_emb = get_text_emb(sd["tokenizer"], sd["text_enc"], "")

    # ── 5. Decode original embedding ──────────────────────────────────────────
    print(f"Decoding original embedding ({num_steps} steps)...")
    gen_orig = decode_embedding(emb_orig, adapter, vae, unet, scheduler, text_emb,
                                cond_scale=cond_scale, num_steps=num_steps, seed=seed)

    # ── 6. Decode compressed → refined embedding ──────────────────────────────
    print(f"Decoding compressed+refined embedding ({num_steps} steps)...")
    gen_comp = decode_embedding(emb_compressed, adapter, vae, unet, scheduler, text_emb,
                                cond_scale=cond_scale, num_steps=num_steps, seed=seed)

    # ── 7. Embedding fidelity metrics ─────────────────────────────────────────
    cos_sim = torch.nn.functional.cosine_similarity(
        emb_orig.reshape(-1, 1024),
        emb_compressed.reshape(-1, 1024),
        dim=-1
    ).mean().item()
    mse = torch.nn.functional.mse_loss(emb_orig, emb_compressed).item()
    print(f"\nEmbedding fidelity — cosine sim: {cos_sim:.4f}  |  MSE: {mse:.6f}")

    # ── 8. Plot ───────────────────────────────────────────────────────────────
    orig_display = orig_pil.resize((384, 384), Image.BICUBIC)

    # Compute amplified absolute difference between the two decoded images
    arr_orig = np.array(gen_orig).astype(np.float32)
    arr_comp = np.array(gen_comp).astype(np.float32)
    diff = np.abs(arr_orig - arr_comp)
    diff_amplified = np.clip(diff * 3, 0, 255).astype(np.uint8)

    fig = plt.figure(figsize=(20, 5))
    gs  = gridspec.GridSpec(1, 4, figure=fig, wspace=0.04)

    titles = [
        "Original",
        f"Adapter → SD\n(original emb, scale={cond_scale})",
        f"Compressor+Refiner → SD\ncos={cos_sim:.3f}  mse={mse:.5f}",
        "Difference (3xamplified)",
    ]
    images = [orig_display, gen_orig, gen_comp, Image.fromarray(diff_amplified)]

    for i, (img_disp, title) in enumerate(zip(images, titles)):
        ax = fig.add_subplot(gs[i])
        ax.imshow(np.array(img_disp))
        ax.set_title(title, fontsize=10, pad=6)
        ax.axis("off")

    fig.suptitle(f"{img_path.name}  |  steps={num_steps}  seed={seed}",
                 fontsize=11, y=1.01)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches="tight")
        print(f"Saved to {save_path}")

    plt.show()
    plt.close()

    return {
        "image_path":     img_path,
        "emb_orig":       emb_orig,
        "emb_compressed": emb_compressed,
        "gen_orig":       gen_orig,
        "gen_comp":       gen_comp,
        "cosine_sim":     cos_sim,
        "mse":            mse,
    }

In [ ]:
chosen_image = "experiments/rocks.jpg"
# ── Config ────────────────────────────────────────────────────────────────────
IMAGE_FOLDER    = "experiments/"
COMPRESSOR_CKPT = "best_compressor.pth"
REFINER_CKPT    = "best_refiner.pth"
COND_SCALE      = 1.0
NUM_STEPS       = 20
SEED            = 42
# ─────────────────────────────────────────────────────────────────────────────

# ── RUN ───────────────────────────────────────────────────────────────────────
results = run_triple_comparison(
    image_folder    = IMAGE_FOLDER,
    image_path      = chosen_image,
    compressor_ckpt = COMPRESSOR_CKPT,
    refiner_ckpt   = REFINER_CKPT,
    cond_scale      = COND_SCALE,
    num_steps       = NUM_STEPS,
    seed            = SEED,
    save_path       = "experiments/outputs/triple_comparison.png",
)

In [ ]:
import cv2
from pathlib import Path
from PIL import Image
import torch
import numpy as np

@torch.no_grad()
def run_video_reconstruction(video_path, compressor_ckpt, refiner_ckpt, output_path, cond_scale, num_steps, seed, fps, max_frames):
    video_path  = Path(video_path)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # ── 1. Open video, grab metadata ─────────────────────────────────────────
    cap = cv2.VideoCapture(str(video_path))
    assert cap.isOpened(), f"Could not open video: {video_path}"

    source_fps    = cap.get(cv2.CAP_PROP_FPS)
    total_frames  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    out_fps       = fps if fps is not None else source_fps
    n_frames      = total_frames if max_frames is None else min(max_frames, total_frames)

    print(f"Video: {video_path.name}")
    print(f"Frames: {n_frames}/{total_frames}  |  FPS: {source_fps:.2f} → {out_fps:.2f}")

    # ── 2. Load compressor + refiner once, keep on GPU throughout ────────────
    print("Loading compressor + refiner...")
    compressor, refiner = load_compressor_refiner(compressor_ckpt, refiner_ckpt)

    # ── 3. Load SD + adapter once ─────────────────────────────────────────────
    print("Loading SD + adapter...")
    sd        = load_sd()
    adapter   = load_adapter()
    vae       = sd["vae"]
    unet      = sd["unet"]
    scheduler = sd["scheduler"]
    text_emb  = get_text_emb(sd["tokenizer"], sd["text_enc"], "")

    # ── 4. Set up video writer (384x384 output) ───────────────────────────────
    writer = cv2.VideoWriter(
        str(output_path),
        cv2.VideoWriter_fourcc(*"mp4v"),
        out_fps,
        (384, 384),
    )
    assert writer.isOpened(), f"Could not open video writer: {output_path}"

    # ── 5. Process frame by frame ─────────────────────────────────────────────
    for i in range(n_frames):
        ret, frame_bgr = cap.read()
        if not ret:
            print(f"  Warning: could not read frame {i}, stopping early.")
            break

        # OpenCV → PIL
        frame_pil = Image.fromarray(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))

        # Encode
        emb = encode_frame(frame_pil)                               # (576, 1024) CPU

        # Compress → refine
        emb_gpu        = emb.unsqueeze(0).to(DEVICE)                # (1, 576, 1024)
        compressed     = compressor(emb_gpu)                        # (1, 144, 1024)
        emb_refined    = refiner(compressed).squeeze(0).cpu()       # (576, 1024)

        # Decode
        decoded_pil = decode_embedding(
            emb_refined, adapter, vae, unet, scheduler, text_emb,
            cond_scale=cond_scale, num_steps=num_steps, seed=seed,
        )

        # PIL → OpenCV BGR → write
        frame_out = cv2.cvtColor(np.array(decoded_pil), cv2.COLOR_RGB2BGR)
        writer.write(frame_out)

        if (i + 1) % 10 == 0 or (i + 1) == n_frames:
            print(f"  [{i+1}/{n_frames}] frames processed")

    # ── 6. Cleanup ────────────────────────────────────────────────────────────
    cap.release()
    writer.release()
    del compressor, refiner
    torch.cuda.empty_cache()

    print(f"\nDone. Saved to {output_path}")
    return output_path

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
VIDEO_PATH      = "experiments/robo.mp4"   # input video
COMPRESSOR_CKPT = "best_compressor.pth"
REFINER_CKPT    = "best_refiner.pth"
OUTPUT_PATH     = "experiments/outputs/reconstructed.mp4"
COND_SCALE      = 1.0
NUM_STEPS       = 8
SEED            = 42
FPS             = 5   # None = match source FPS
MAX_FRAMES      = 60   # None = process entire video, or set e.g. 60 to cap it
# ─────────────────────────────────────────────────────────────────────────────

run_video_reconstruction(video_path=VIDEO_PATH,
                         compressor_ckpt=COMPRESSOR_CKPT,
                         refiner_ckpt=REFINER_CKPT,
                         output_path=OUTPUT_PATH,
                         cond_scale=COND_SCALE,
                         num_steps=NUM_STEPS,
                         seed=SEED,
                         fps=FPS,
                         max_frames=MAX_FRAMES)